# EDA — Popularity & Ratings Features

This notebook explores the two community-signal features used in the Mixtape recommendation engine:

| Feature | Source | Matrix | Columns |
|---|---|---|---|
| **Popularity** | Last.fm scraped data | `album_lastfm_popularity_matrix.npz` | album_listeners, album_scrobbles, artist_listeners, artist_scrobbles |
| **Ratings** | MusicBrainz user ratings | `album_ratings_matrix.npz` | Bayesian-weighted score (0–1) |

Both features reflect how well-known or well-received an album is in the community — but from different angles:
- **Popularity** = how many people played it (Last.fm play counts)
- **Ratings** = how well people rated it (MusicBrainz star ratings, Bayesian-weighted)

In the app these two are **synced** — the Popularity knob controls both simultaneously.

In [ ]:
import os, re, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz

DATA_DIR     = '../data'
FEATURES_DIR = '../data/features'

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

---
## Part 1 — Last.fm Popularity Feature

### How it was built

1. Scraped Last.fm artist and album pages for listeners and scrobble counts
2. Saved to `data/lastfm_data.parquet`
3. Matched to MusicBrainz `album_id` by normalised (artist, album) name
4. Min-max scaled each of the 4 signals to [0, 1]
5. Built a sparse matrix — same row order as `album_ids.pkl`

In [ ]:
# ── Load raw scraped data ────────────────────────────────────────────────────
df = pd.read_parquet(os.path.join(DATA_DIR, 'lastfm_data.parquet'))

# Normalise numeric columns
for col in ['Artist_Listeners', 'Artist_Scrobbles', 'Album_Listeners', 'Album_Scrobbles']:
    df[col] = (
        df[col].astype(str)
        .str.replace(',', '', regex=False).str.strip()
        .replace({'N/A': np.nan, 'nan': np.nan, 'None Found': np.nan, '': np.nan})
        .astype(float)
    )

print(f'Total scraped albums : {len(df):,}')
print(f'Unique artists       : {df["Artist"].nunique():,}')
print(f'Null Album_Listeners : {df["Album_Listeners"].isna().sum():,} ({100*df["Album_Listeners"].isna().mean():.1f}%)')
print()
df.describe()

In [ ]:
# ── Distribution of Album Scrobbles ─────────────────────────────────────────
df_valid = df.dropna(subset=['Album_Scrobbles'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Log scale — full range
axes[0].hist(df_valid['Album_Scrobbles'][df_valid['Album_Scrobbles'] > 0],
             bins=80, color='#4A90E2', edgecolor='none', log=True)
axes[0].set_title('Album Scrobbles Distribution (log scale)', fontsize=13, weight='bold')
axes[0].set_xlabel('Scrobbles')
axes[0].set_ylabel('Count (log)')

# Top 20 most scrobbled albums
top20 = df_valid.nlargest(20, 'Album_Scrobbles')[['Artist', 'Album', 'Album_Scrobbles']]
axes[1].barh(range(20), top20['Album_Scrobbles'].values, color='#E056FD')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f"{r.Artist} — {r.Album}" for _, r in top20.iterrows()], fontsize=8)
axes[1].invert_yaxis()
axes[1].set_title('Top 20 Albums by Scrobbles', fontsize=13, weight='bold')
axes[1].set_xlabel('Album Scrobbles')

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution of Artist Listeners ────────────────────────────────────────
df_valid_a = df.dropna(subset=['Artist_Listeners']).drop_duplicates(subset=['Artist'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df_valid_a['Artist_Listeners'][df_valid_a['Artist_Listeners'] > 0],
             bins=80, color='#10AC84', edgecolor='none', log=True)
axes[0].set_title('Artist Listeners Distribution (log scale)', fontsize=13, weight='bold')
axes[0].set_xlabel('Listeners')
axes[0].set_ylabel('Count (log)')

top20a = df_valid_a.nlargest(20, 'Artist_Listeners')[['Artist', 'Artist_Listeners']]
axes[1].barh(range(20), top20a['Artist_Listeners'].values, color='#FF6B6B')
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20a['Artist'].values, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_title('Top 20 Artists by Listeners', fontsize=13, weight='bold')
axes[1].set_xlabel('Artist Listeners')

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation between the 4 signals ───────────────────────────────────────
df_corr = df[['Album_Listeners', 'Album_Scrobbles',
              'Artist_Listeners', 'Artist_Scrobbles']].dropna()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Correlation between Popularity Signals', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  High correlation between listeners and scrobbles = expected (more fans = more plays)')
print('  Album signals vs artist signals shows how album popularity relates to artist fame')

In [ ]:
# ── Coverage — how many of the 1.75M albums have Last.fm data ────────────────
with open(os.path.join(FEATURES_DIR, 'album_ids.pkl'), 'rb') as f:
    album_ids = np.asarray(pickle.load(f))

pop_mat = load_npz(os.path.join(FEATURES_DIR, 'album_lastfm_popularity_matrix.npz'))
coverage = int((pop_mat.sum(axis=1) > 0).sum())

print(f'Total albums in universe : {len(album_ids):,}')
print(f'Albums with Last.fm data : {coverage:,}  ({100*coverage/len(album_ids):.2f}%)')
print(f'Albums with no data      : {len(album_ids)-coverage:,}  (score = 0 in matrix)')
print(f'Matrix shape             : {pop_mat.shape}')
print(f'Matrix NNZ               : {pop_mat.nnz:,}')
print()

# Columns of the matrix
print('Matrix columns:')
for i, name in enumerate(['album_listeners', 'album_scrobbles', 'artist_listeners', 'artist_scrobbles']):
    col = np.asarray(pop_mat[:, i].todense()).ravel()
    nonzero = (col > 0).sum()
    print(f'  col {i}: {name:25s}  non-zero: {nonzero:,}  max: {col.max():.4f}  mean(non-zero): {col[col>0].mean():.4f}')

---
## Part 2 — MusicBrainz Ratings Feature

### How it was built

MusicBrainz stores user-submitted ratings (0–100) for albums and artists.
The raw rating is unreliable for sparse data — an album with 1 rating of 100 looks better than one with 500 ratings of 95.

**Solution: Zero-anchored Bayesian weighting**

```
weighted_score = (R × v) / (v + C)

R = mean rating (0–100)
v = number of votes
C = 5  (confidence constant)
```

- When `v = 0` → score = 0 (no data = no signal)
- When `v = 5` → score = R/2 (half weight)
- When `v → ∞` → score → R (full trust)

Then divided by 100 to normalise to [0, 1].

In [ ]:
# ── Load MusicBrainz ratings ─────────────────────────────────────────────────
album_ratings  = pd.read_parquet(os.path.join(DATA_DIR, 'mb_album_ratings.parquet'))
artist_ratings = pd.read_parquet(os.path.join(DATA_DIR, 'mb_artist_ratings.parquet'))

print(f'Album ratings  : {len(album_ratings):,} rows')
print(f'Artist ratings : {len(artist_ratings):,} rows')
print()
print('Album ratings sample:')
display(album_ratings.head(5))

In [ ]:
# ── Apply Bayesian weighting ─────────────────────────────────────────────────
C = 5

for df_r, name in [(album_ratings, 'Album'), (artist_ratings, 'Artist')]:
    v = df_r['rating_count'].fillna(0)
    R = df_r['rating'].fillna(0)
    df_r['weighted_score']      = (R * v) / (v + C)
    df_r['weighted_score_norm'] = df_r['weighted_score'] / 100.0
    print(f'{name} ratings — weighted_score_norm stats:')
    print(df_r['weighted_score_norm'].describe().round(4))
    print()

In [ ]:
# ── Visualise: effect of Bayesian weighting ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw rating distribution
sns.histplot(album_ratings['rating'].dropna(), bins=50, ax=axes[0], color='#4A90E2', kde=True)
axes[0].set_title('Raw Album Ratings (0–100)', fontsize=12, weight='bold')
axes[0].set_xlabel('Rating')

# Bayesian score — all albums (log scale because ~95% are 0)
sns.histplot(album_ratings['weighted_score_norm'], bins=60, ax=axes[1], color='#E056FD', kde=False, log_scale=(False, True))
axes[1].set_title('Bayesian Score — All Albums (log y)', fontsize=12, weight='bold')
axes[1].set_xlabel('Weighted Score (0–1)')

# Bayesian score — rated albums only
rated = album_ratings[album_ratings['weighted_score_norm'] > 0]
sns.histplot(rated['weighted_score_norm'], bins=50, ax=axes[2], color='#10AC84', kde=True)
axes[2].set_title(f'Bayesian Score — Rated Only (n={len(rated):,})', fontsize=12, weight='bold')
axes[2].set_xlabel('Weighted Score (0–1)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Why C=5? Show effect of different C values ───────────────────────────────
example_R = 90   # an album rated 90/100
votes = np.arange(0, 51)

fig, ax = plt.subplots(figsize=(10, 5))
for C_val, color, label in [(1, '#FF6B6B', 'C=1 (too aggressive)'),
                             (5, '#10AC84', 'C=5 (used in model)'),
                             (25, '#4A90E2', 'C=25 (too conservative)')]:
    scores = (example_R * votes) / (votes + C_val) / 100
    ax.plot(votes, scores, label=label, color=color, linewidth=2)

ax.axhline(y=0.9, color='grey', linestyle='--', linewidth=1, label='Raw rating (0.90)')
ax.set_xlabel('Number of votes', fontsize=11)
ax.set_ylabel('Bayesian score (normalised)', fontsize=11)
ax.set_title(f'Effect of C on Bayesian Score (R={example_R}/100)', fontsize=13, weight='bold')
ax.legend()
ax.set_xlim(0, 50)
plt.tight_layout()
plt.show()

print('At C=5: an album needs ~5 votes to reach 50% of its raw score.')
print('This prevents a single 5-star rating from dominating.')

In [ ]:
# ── Ratings coverage in the matrix ───────────────────────────────────────────
rat_mat = load_npz(os.path.join(FEATURES_DIR, 'album_ratings_matrix.npz'))
rat_coverage = int((rat_mat.sum(axis=1) > 0).sum())

print(f'Ratings matrix shape   : {rat_mat.shape}')
print(f'Albums with ratings    : {rat_coverage:,}  ({100*rat_coverage/len(album_ids):.2f}%)')
print(f'Albums with no rating  : {len(album_ids)-rat_coverage:,}')
print()

scores = np.asarray(rat_mat.todense()).ravel()
top10 = np.argsort(-scores)[:10]

lookup = (
    pd.read_parquet(os.path.join(DATA_DIR, 'mb_album_artists.parquet'),
                    columns=['album_id', 'album_name', 'artist_name'])
    .drop_duplicates(subset='album_id').set_index('album_id')
)
print('Top 10 albums by Bayesian rating score:')
for i, idx in enumerate(top10):
    aid = int(album_ids[idx])
    r = lookup.loc[aid] if aid in lookup.index else {'album_name': '?', 'artist_name': '?'}
    print(f'  {i+1:2d}. {r["artist_name"]} — {r["album_name"]}  ({scores[idx]:.4f})')

---
## Part 3 — Popularity vs Ratings: How they compare

In [ ]:
# ── Coverage comparison ───────────────────────────────────────────────────────
total = len(album_ids)

fig, ax = plt.subplots(figsize=(8, 4))
features = ['Popularity (Last.fm)', 'Ratings (MusicBrainz)']
covered  = [coverage, rat_coverage]
colors   = ['#4A90E2', '#E056FD']

bars = ax.barh(features, [c/total*100 for c in covered], color=colors)
for bar, n in zip(bars, covered):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{n:,} albums ({n/total*100:.1f}%)', va='center', fontsize=10)
ax.set_xlabel('% of albums covered')
ax.set_title('Coverage: % of 1.75M albums with signal', fontsize=13, weight='bold')
ax.set_xlim(0, 35)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: do popular albums also get good ratings? ────────────────────────
# Find albums present in both datasets
pop_scores = np.asarray(pop_mat[:, 1].todense()).ravel()   # album scrobbles
rat_scores = np.asarray(rat_mat.todense()).ravel()

both_mask = (pop_scores > 0) & (rat_scores > 0)
print(f'Albums in both datasets: {both_mask.sum():,}')

sample = np.random.choice(np.where(both_mask)[0], size=min(5000, both_mask.sum()), replace=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pop_scores[sample], rat_scores[sample],
           alpha=0.3, s=10, color='#4A90E2')
ax.set_xlabel('Popularity score (album scrobbles, scaled)', fontsize=11)
ax.set_ylabel('Rating score (Bayesian, normalised)', fontsize=11)
ax.set_title('Popularity vs Ratings (albums in both datasets)', fontsize=13, weight='bold')

corr = np.corrcoef(pop_scores[sample], rat_scores[sample])[0, 1]
ax.text(0.05, 0.92, f'Pearson r = {corr:.3f}', transform=ax.transAxes,
        fontsize=11, color='#E056FD', weight='bold')
plt.tight_layout()
plt.show()

print(f'Correlation: {corr:.3f}')
print('This is why the app syncs ratings weight to the popularity knob.')

---
## Summary

| | Popularity | Ratings |
|---|---|---|
| **Source** | Last.fm (scraped) | MusicBrainz (database) |
| **Signal** | Play counts & listeners | Star ratings (Bayesian weighted) |
| **Coverage** | ~14% of albums | ~5% of albums |
| **Matrix** | 1,758,488 × 4 | 1,758,488 × 1 |
| **Scale** | Min-max [0, 1] | Bayesian ÷ 100 → [0, 1] |
| **In app** | Popularity knob | Synced to Popularity knob |

**Key insight:** Popular albums tend to also be well-rated (positive correlation). Syncing both features to a single knob gives the user a single intuitive control: *"How much should fame/reputation influence my recommendations?"*